In [1]:
#CONNECT TO DATABASE

In [129]:
import duckdb
import pandas as pd
from pathlib import Path

# DB path
DB_PATH = "data/market_data.duckdb"

# Connect
con = duckdb.connect(DB_PATH)

print("✅ Connected to DuckDB")


✅ Connected to DuckDB


In [3]:
#CLEAN RESET (DROP OLD OBJECTS SAFELY)

In [4]:
import duckdb

DB_PATH = "data/market_data.duckdb"
con = duckdb.connect(DB_PATH)

# Objects we want to remove (name only, no type assumptions)
objects_to_drop = [
    "market_prices_daily",
    "stock_prices_daily",
    "market_returns_daily",
    "stock_returns_daily",
    "market_return_annualised_correct",
    "stock_return_annualised_correct",
    "beta_2y_calculated",
    "market_erp_correct",
    "cost_of_equity_latest",
]

# Drop TABLES if they exist
tables = set(
    con.execute("SELECT table_name FROM duckdb_tables()").fetchall()
)
tables = {t[0] for t in tables}

# Drop VIEWS if they exist
views = set(
    con.execute("SELECT view_name FROM duckdb_views()").fetchall()
)
views = {v[0] for v in views}

for obj in objects_to_drop:
    if obj in tables:
        con.execute(f"DROP TABLE {obj}")
        print(f"🗑️ Dropped TABLE: {obj}")
    elif obj in views:
        con.execute(f"DROP VIEW {obj}")
        print(f"🗑️ Dropped VIEW : {obj}")

print("✅ Database reset completed safely")


✅ Database reset completed safely


In [5]:
#CLEAN RESET (DROP OLD OBJECTS SAFELY)

In [6]:
BASE_PATH = Path(r"C:\Users\Ojasvi Vij\Documents\NIFTY500 Agent\data")

NIFTY_FILES = [
    BASE_PATH / "NIFTY 500-01-01-2023-to-31-12-2023.csv",
    BASE_PATH / "NIFTY 500-01-01-2024-to-31-12-2024.csv",
    BASE_PATH / "NIFTY 500-01-01-2025-to-31-12-2025.csv",
    BASE_PATH / "NIFTY 500-01-01-2026-to-07-02-2026.csv",
]

STOCK_PRICE_FILE = BASE_PATH / "data_daily.xlsx"


In [7]:
#CELL 3 — LOAD NIFTY 500 DAILY PRICES

In [8]:
dfs = []

for f in NIFTY_FILES:
    df = pd.read_csv(f)
    df.columns = df.columns.str.lower().str.strip()
    dfs.append(df)

nifty = pd.concat(dfs, ignore_index=True)

nifty = nifty.rename(columns={
    "date": "date",
    "close": "adj_close"
})

nifty["date"] = pd.to_datetime(nifty["date"])

con.execute("""
CREATE TABLE market_prices_daily (
    date DATE,
    adj_close DOUBLE
)
""")

con.register("nifty_df", nifty[["date", "adj_close"]])
con.execute("INSERT INTO market_prices_daily SELECT * FROM nifty_df")

print("✅ Market prices loaded")


✅ Market prices loaded


C:\Users\Ojasvi Vij\AppData\Local\Temp\ipykernel_33024\3449762393.py:15: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  nifty["date"] = pd.to_datetime(nifty["date"])


In [9]:
#CREATE MARKET DAILY RETURNS (LOG RETURNS)

In [10]:
con.execute("""
CREATE VIEW market_returns_daily AS
SELECT
    date,
    LN(adj_close / LAG(adj_close) OVER (ORDER BY date)) AS market_return
FROM market_prices_daily
""")

print("✅ Market daily returns created")


✅ Market daily returns created


In [11]:
con.execute("SELECT * FROM market_returns_daily LIMIT 5").df()

,date,market_return
0,2023-01-02,NaN
1,2023-01-03,0.002197
2,2023-01-04,-0.010649
3,2023-01-05,-0.000442
4,2023-01-06,-0.007450


In [12]:
#CORRECT ANNUALISED MARKET RETURN (GEOMETRIC)

In [13]:
con.execute("""
CREATE TABLE market_return_annualised_correct AS
SELECT
    MIN(date) AS start_date,
    MAX(date) AS end_date,

    EXP(SUM(market_return)) - 1 AS total_return,

    DATE_DIFF('day', MIN(date), MAX(date)) / 365.25 AS years,

    POWER(
        EXP(SUM(market_return)),
        1.0 / (DATE_DIFF('day', MIN(date), MAX(date)) / 365.25)
    ) - 1 AS annualised_return
FROM market_returns_daily
""")

print("✅ Annualised market return computed")


✅ Annualised market return computed


In [14]:
con.execute("SELECT * FROM market_return_annualised_correct").df()

,start_date,end_date,total_return,years,annualised_return
0,2023-01-02,2026-02-06,0.509568,3.096509,0.142245


In [15]:
#LOAD STOCK DAILY PRICES

In [16]:
df = pd.read_excel(STOCK_PRICE_FILE)
df.columns = df.columns.str.lower().str.strip()

df = df.rename(columns={"close": "adj_close"})
df["date"] = pd.to_datetime(df["date"])

stock_map = dict(
    con.execute("SELECT symbol, stock_id FROM stocks_master").fetchall()
)

df["stock_id"] = df["symbol"].map(stock_map)
df = df.dropna(subset=["stock_id"])

con.execute("""
CREATE TABLE stock_prices_daily (
    stock_id INTEGER,
    date DATE,
    adj_close DOUBLE
)
""")

con.register("prices_df", df[["stock_id", "date", "adj_close"]])
con.execute("INSERT INTO stock_prices_daily SELECT * FROM prices_df")

print("✅ Stock prices loaded")


✅ Stock prices loaded


In [17]:
#STOCK DAILY RETURNS (LOG RETURNS)

In [18]:
con.execute("""
CREATE VIEW stock_returns_daily AS
SELECT
    stock_id,
    date,
    LN(adj_close / LAG(adj_close) OVER (
        PARTITION BY stock_id ORDER BY date
    )) AS stock_return
FROM stock_prices_daily
""")

print("✅ Stock daily returns created")


✅ Stock daily returns created


In [19]:
#CORRECT ANNUALISED STOCK RETURNS

In [20]:
con.execute("""
CREATE TABLE stock_return_annualised_correct AS
SELECT
    stock_id,

    MIN(date) AS start_date,
    MAX(date) AS end_date,

    EXP(SUM(stock_return)) - 1 AS total_return,

    DATE_DIFF('day', MIN(date), MAX(date)) / 365.25 AS years,

    POWER(
        EXP(SUM(stock_return)),
        1.0 / (DATE_DIFF('day', MIN(date), MAX(date)) / 365.25)
    ) - 1 AS annualised_return
FROM stock_returns_daily
GROUP BY stock_id
""")

print("✅ Annualised stock returns computed")


✅ Annualised stock returns computed


In [21]:
#ROLLING BETA (2 YEARS ≈ 504 DAYS)

In [22]:
con.execute("""
CREATE TABLE beta_2y_calculated AS
SELECT
    s.stock_id,
    s.date,

    COVAR_SAMP(s.stock_return, m.market_return)
        OVER w
    /
    VAR_SAMP(m.market_return)
        OVER w AS beta_2y
FROM stock_returns_daily s
JOIN market_returns_daily m USING (date)
WINDOW w AS (
    PARTITION BY s.stock_id
    ORDER BY s.date
    ROWS BETWEEN 503 PRECEDING AND CURRENT ROW
)
""")

print("✅ Rolling beta calculated")


✅ Rolling beta calculated


In [23]:
#COST OF EQUITY (USING REALISED ERP)

In [24]:
RISK_FREE = 0.072

con.execute(f"""
CREATE TABLE cost_of_equity_latest AS
WITH erp AS (
    SELECT
        annualised_return - {RISK_FREE} AS equity_risk_premium
    FROM market_return_annualised_correct
)
SELECT
    b.stock_id,
    b.date,
    b.beta_2y,
    erp.equity_risk_premium,
    {RISK_FREE} + b.beta_2y * erp.equity_risk_premium AS cost_of_equity
FROM (
    SELECT *,
           ROW_NUMBER() OVER (
               PARTITION BY stock_id
               ORDER BY date DESC
           ) AS rn
    FROM beta_2y_calculated
) b
CROSS JOIN erp
WHERE rn = 1
""")

print("✅ Cost of Equity (realised ERP) created")


✅ Cost of Equity (realised ERP) created


In [25]:
con.execute("""
SELECT
    sm.symbol,
    c.beta_2y,
    c.equity_risk_premium,
    c.cost_of_equity
FROM cost_of_equity_latest c
JOIN stocks_master sm USING (stock_id)
ORDER BY sm.symbol
LIMIT 20
""").df()

,symbol,beta_2y,equity_risk_premium,cost_of_equity
0,360ONE,1.095793,0.070245,0.148974
1,3MINDIA,0.563082,0.070245,0.111554
2,AADHARHFC,0.838008,0.070245,0.130866
3,AARTIIND,1.480084,0.070245,0.175969
4,AAVAS,0.702010,0.070245,0.121313
5,ABB,1.353011,0.070245,0.167043
6,ABBOTINDIA,0.421347,0.070245,0.101598
7,ABCAPITAL,1.489559,0.070245,0.176635
8,ABFRL,1.566235,0.070245,0.182021
9,ABREL,1.590060,0.070245,0.183694


In [26]:
con.execute("""
CREATE OR REPLACE VIEW vw_debt_march AS
SELECT
    stock_id,
    period_end,
    total_debt
FROM balance_sheet_calculated
WHERE EXTRACT(month FROM period_end) = 3;
""")


In [27]:
con.execute("""
SELECT
    sm.symbol,
    d.period_end,
    d.total_debt
FROM vw_debt_march d
JOIN stocks_master sm
  ON d.stock_id = sm.stock_id
WHERE sm.symbol = 'RELIANCE'
ORDER BY d.period_end;
""").df()


,symbol,period_end,total_debt
0,RELIANCE,2014-03-01,138761.0
1,RELIANCE,2015-03-01,168251.0
2,RELIANCE,2016-03-01,194714.0
3,RELIANCE,2017-03-01,217475.0
4,RELIANCE,2018-03-01,239843.0
5,RELIANCE,2019-03-01,307714.0
6,RELIANCE,2020-03-01,355133.0
7,RELIANCE,2021-03-01,278962.0
8,RELIANCE,2022-03-01,319158.0
9,RELIANCE,2023-03-01,451664.0


In [28]:
con.execute("""
CREATE OR REPLACE VIEW vw_interest_march AS
SELECT
    stock_id,
    period_end,
    interest_expense
FROM income_statement_calculated
WHERE EXTRACT(month FROM period_end) = 3;
""")


In [29]:
con.execute("""
SELECT
    sm.symbol,
    i.period_end,
    i.interest_expense
FROM vw_interest_march i
JOIN stocks_master sm
  ON i.stock_id = sm.stock_id
WHERE sm.symbol = 'RELIANCE'
ORDER BY i.period_end;
""").df()


,symbol,period_end,interest_expense
0,RELIANCE,2014-03-01,3836.0
1,RELIANCE,2015-03-01,3316.0
2,RELIANCE,2016-03-01,3691.0
3,RELIANCE,2017-03-01,3849.0
4,RELIANCE,2018-03-01,8052.0
5,RELIANCE,2019-03-01,16495.0
6,RELIANCE,2020-03-01,22027.0
7,RELIANCE,2021-03-01,21189.0
8,RELIANCE,2022-03-01,14584.0
9,RELIANCE,2023-03-01,19571.0


In [30]:
con.execute("""
DROP TABLE IF EXISTS tbl_avg_debt;

CREATE TABLE tbl_avg_debt AS
SELECT
    stock_id,
    period_end,
    total_debt,
    (
        total_debt
      + LAG(total_debt) OVER (
            PARTITION BY stock_id
            ORDER BY period_end
        )
    ) / 2.0 AS avg_debt
FROM vw_debt_march;
""")


In [31]:
con.execute("""
SELECT
    sm.symbol,
    d.period_end,
    d.total_debt,
    d.avg_debt
FROM tbl_avg_debt d
JOIN stocks_master sm
  ON d.stock_id = sm.stock_id
WHERE sm.symbol = 'RELIANCE'
ORDER BY d.period_end;
""").df()


,symbol,period_end,total_debt,avg_debt
0,RELIANCE,2014-03-01,138761.0,NaN
1,RELIANCE,2015-03-01,168251.0,153506.0
2,RELIANCE,2016-03-01,194714.0,181482.5
3,RELIANCE,2017-03-01,217475.0,206094.5
4,RELIANCE,2018-03-01,239843.0,228659.0
5,RELIANCE,2019-03-01,307714.0,273778.5
6,RELIANCE,2020-03-01,355133.0,331423.5
7,RELIANCE,2021-03-01,278962.0,317047.5
8,RELIANCE,2022-03-01,319158.0,299060.0
9,RELIANCE,2023-03-01,451664.0,385411.0


In [32]:
con.execute("""
DROP TABLE IF EXISTS tbl_cost_of_debt;

CREATE TABLE tbl_cost_of_debt AS
SELECT
    d.stock_id,
    d.period_end,
    d.avg_debt,
    i.interest_expense,
    CASE
        WHEN d.avg_debt IS NULL OR d.avg_debt = 0 THEN NULL
        ELSE i.interest_expense / d.avg_debt
    END AS cost_of_debt_pre_tax
FROM tbl_avg_debt d
LEFT JOIN vw_interest_march i
  ON d.stock_id = i.stock_id
 AND d.period_end = i.period_end;
""")


In [33]:
con.execute("""
SELECT
    sm.symbol,
    c.period_end,
    c.avg_debt,
    c.interest_expense,
    ROUND(c.cost_of_debt_pre_tax * 100, 2) AS cost_of_debt_pct
FROM tbl_cost_of_debt c
JOIN stocks_master sm
  ON c.stock_id = sm.stock_id
WHERE sm.symbol = 'RELIANCE'
ORDER BY c.period_end;
""").df()


,symbol,period_end,avg_debt,interest_expense,cost_of_debt_pct
0,RELIANCE,2014-03-01,NaN,3836.0,NaN
1,RELIANCE,2015-03-01,153506.0,3316.0,2.16
2,RELIANCE,2016-03-01,181482.5,3691.0,2.03
3,RELIANCE,2017-03-01,206094.5,3849.0,1.87
4,RELIANCE,2018-03-01,228659.0,8052.0,3.52
5,RELIANCE,2019-03-01,273778.5,16495.0,6.02
6,RELIANCE,2020-03-01,331423.5,22027.0,6.65
7,RELIANCE,2021-03-01,317047.5,21189.0,6.68
8,RELIANCE,2022-03-01,299060.0,14584.0,4.88
9,RELIANCE,2023-03-01,385411.0,19571.0,5.08


In [34]:
con.execute("""
DROP TABLE IF EXISTS tbl_cost_of_debt_latest;

CREATE TABLE tbl_cost_of_debt_latest AS
SELECT *
FROM (
    SELECT
        stock_id,
        period_end,
        cost_of_debt_pre_tax,
        ROW_NUMBER() OVER (
            PARTITION BY stock_id
            ORDER BY period_end DESC
        ) AS rn
    FROM tbl_cost_of_debt
    WHERE cost_of_debt_pre_tax IS NOT NULL
) t
WHERE rn = 1;
""")


In [35]:
con.execute("""
SELECT
    sm.symbol,
    l.period_end,
    ROUND(l.cost_of_debt_pre_tax * 100, 2) AS cost_of_debt_pct
FROM tbl_cost_of_debt_latest l
JOIN stocks_master sm
  ON l.stock_id = sm.stock_id
WHERE sm.symbol = 'RELIANCE';
""").df()


,symbol,period_end,cost_of_debt_pct
0,RELIANCE,2025-03-01,6.69


In [36]:
#TOTAL EQUITY (SOURCE, NOT RECOMPUTED

In [37]:
con.execute("""
CREATE OR REPLACE VIEW vw_equity_march AS
SELECT
    stock_id,
    period_end,
    total_equity
FROM balance_sheet_calculated
WHERE EXTRACT(month FROM period_end) = 3;
""")


In [38]:
con.execute("""
SELECT
    sm.symbol,
    e.period_end,
    e.total_equity
FROM vw_equity_march e
JOIN stocks_master sm
  ON e.stock_id = sm.stock_id
WHERE sm.symbol = 'RELIANCE'
ORDER BY e.period_end;
""").df()


,symbol,period_end,total_equity
0,RELIANCE,2014-03-01,198687.0
1,RELIANCE,2015-03-01,218499.0
2,RELIANCE,2016-03-01,231556.0
3,RELIANCE,2017-03-01,263709.0
4,RELIANCE,2018-03-01,293506.0
5,RELIANCE,2019-03-01,387112.0
6,RELIANCE,2020-03-01,449166.0
7,RELIANCE,2021-03-01,700172.0
8,RELIANCE,2022-03-01,779485.0
9,RELIANCE,2023-03-01,715872.0


In [39]:
#NET DEBT (SOURCE, NOT RECOMPUTED)

In [40]:
con.execute("""
CREATE OR REPLACE VIEW vw_net_debt_march AS
SELECT
    stock_id,
    period_end,
    net_debt
FROM balance_sheet_calculated
WHERE EXTRACT(month FROM period_end) = 3;
""")


In [41]:
con.execute("""
SELECT
    sm.symbol,
    n.period_end,
    n.net_debt
FROM vw_net_debt_march n
JOIN stocks_master sm
  ON n.stock_id = sm.stock_id
WHERE sm.symbol = 'RELIANCE'
ORDER BY n.period_end;
""").df()


,symbol,period_end,net_debt
0,RELIANCE,2014-03-01,100777.0
1,RELIANCE,2015-03-01,155706.0
2,RELIANCE,2016-03-01,183686.0
3,RELIANCE,2017-03-01,214452.0
4,RELIANCE,2018-03-01,235588.0
5,RELIANCE,2019-03-01,296633.0
6,RELIANCE,2020-03-01,324213.0
7,RELIANCE,2021-03-01,261565.0
8,RELIANCE,2022-03-01,282980.0
9,RELIANCE,2023-03-01,383000.0


In [42]:
con.execute("""
DROP TABLE IF EXISTS tbl_capital_structure;

CREATE TABLE tbl_capital_structure AS
SELECT
    d.stock_id,
    d.period_end,
    d.total_debt,
    e.total_equity,

    (d.total_debt + e.total_equity) AS total_capital,

    CASE
        WHEN (d.total_debt + e.total_equity) = 0 THEN NULL
        ELSE d.total_debt / (d.total_debt + e.total_equity)
    END AS weight_debt,

    CASE
        WHEN (d.total_debt + e.total_equity) = 0 THEN NULL
        ELSE e.total_equity / (d.total_debt + e.total_equity)
    END AS weight_equity

FROM vw_debt_march d
JOIN vw_equity_march e
  ON d.stock_id = e.stock_id
 AND d.period_end = e.period_end;
""")


In [43]:
con.execute("""
CREATE OR REPLACE VIEW vw_capital_structure AS
SELECT * FROM tbl_capital_structure;
""")


In [44]:
con.execute("""
SELECT
    sm.symbol,
    c.period_end,
    c.total_debt,
    c.total_equity,
    ROUND(c.weight_debt * 100, 2)   AS weight_debt_pct,
    ROUND(c.weight_equity * 100, 2) AS weight_equity_pct
FROM vw_capital_structure c
JOIN stocks_master sm
  ON c.stock_id = sm.stock_id
WHERE sm.symbol = 'RELIANCE'
ORDER BY c.period_end;
""").df()


,symbol,period_end,total_debt,total_equity,weight_debt_pct,weight_equity_pct
0,RELIANCE,2014-03-01,138761.0,198687.0,41.12,58.88
1,RELIANCE,2015-03-01,168251.0,218499.0,43.50,56.50
2,RELIANCE,2016-03-01,194714.0,231556.0,45.68,54.32
3,RELIANCE,2017-03-01,217475.0,263709.0,45.20,54.80
4,RELIANCE,2018-03-01,239843.0,293506.0,44.97,55.03
5,RELIANCE,2019-03-01,307714.0,387112.0,44.29,55.71
6,RELIANCE,2020-03-01,355133.0,449166.0,44.15,55.85
7,RELIANCE,2021-03-01,278962.0,700172.0,28.49,71.51
8,RELIANCE,2022-03-01,319158.0,779485.0,29.05,70.95
9,RELIANCE,2023-03-01,451664.0,715872.0,38.69,61.31


In [45]:
con.execute("""
DROP TABLE IF EXISTS tbl_capital_structure_latest;

CREATE TABLE tbl_capital_structure_latest AS
SELECT *
FROM (
    SELECT
        stock_id,
        period_end,
        total_debt,
        total_equity,
        weight_debt,
        weight_equity,
        ROW_NUMBER() OVER (
            PARTITION BY stock_id
            ORDER BY period_end DESC
        ) AS rn
    FROM tbl_capital_structure
) t
WHERE rn = 1;
""")


In [46]:
con.execute("""
SELECT
    sm.symbol,
    c.period_end,
    ROUND(c.weight_debt * 100, 2)   AS weight_debt_pct,
    ROUND(c.weight_equity * 100, 2) AS weight_equity_pct
FROM tbl_capital_structure_latest c
JOIN stocks_master sm
  ON c.stock_id = sm.stock_id
WHERE sm.symbol = 'RELIANCE';
""").df()


,symbol,period_end,weight_debt_pct,weight_equity_pct
0,RELIANCE,2025-03-01,30.74,69.26


In [47]:
TAX_RATE = 0.25

con.execute(f"""
DROP TABLE IF EXISTS tbl_wacc_latest;

CREATE TABLE tbl_wacc_latest AS
SELECT
    cs.stock_id,

    cs.period_end              AS capital_date,
    ce.date                    AS equity_date,

    cs.weight_equity,
    cs.weight_debt,

    ce.cost_of_equity,
    cd.cost_of_debt_pre_tax,

    -- WACC calculation
    (
        cs.weight_equity * ce.cost_of_equity
      + cs.weight_debt
        * cd.cost_of_debt_pre_tax
        * (1 - {TAX_RATE})
    ) AS wacc

FROM tbl_capital_structure_latest cs

LEFT JOIN cost_of_equity_latest ce
  ON cs.stock_id = ce.stock_id

LEFT JOIN tbl_cost_of_debt_latest cd
  ON cs.stock_id = cd.stock_id;
""")

print("✅ WACC (latest) created")


✅ WACC (latest) created


In [48]:
con.execute("""
CREATE OR REPLACE VIEW vw_wacc_latest AS
SELECT * FROM tbl_wacc_latest;
""")


In [659]:
con.execute("""
SELECT
    sm.symbol,

    ROUND(weight_equity * 100, 2) AS weight_equity_pct,
    ROUND(weight_debt * 100, 2)   AS weight_debt_pct,

    ROUND(cost_of_equity * 100, 2)        AS cost_of_equity_pct,
    ROUND(cost_of_debt_pre_tax * 100, 2)  AS cost_of_debt_pct,

    ROUND(wacc * 100, 2) AS wacc_pct
FROM vw_wacc_latest w
JOIN stocks_master sm
  ON w.stock_id = sm.stock_id
WHERE sm.symbol in ('RELIANCE','BEL','ASIANPAINT','*');
""").df()


,symbol,weight_equity_pct,weight_debt_pct,cost_of_equity_pct,cost_of_debt_pct,wacc_pct
0,ASIANPAINT,89.44,10.56,10.69,9.53,10.32
1,RELIANCE,69.26,30.74,14.30,6.69,11.44
2,BEL,99.70,0.30,18.58,20.97,18.57


In [580]:
con.execute("""
-- FCFF & growth
DROP TABLE IF EXISTS tbl_fcff_base;
DROP TABLE IF EXISTS tbl_fcff_normalised;
DROP TABLE IF EXISTS tbl_fcff_sector_calculated;
DROP TABLE IF EXISTS tbl_fcff_driver_forecast;
DROP TABLE IF EXISTS tbl_fcff_driver_pv;
DROP TABLE IF EXISTS tbl_nopat_forecast;
DROP TABLE IF EXISTS tbl_revenue_growth;
DROP TABLE IF EXISTS tbl_growth_final;
DROP TABLE IF EXISTS tbl_revenue_forecast;
DROP TABLE IF EXISTS tbl_driver_assumptions;

-- Terminal & valuation
DROP TABLE IF EXISTS tbl_terminal_growth;
DROP TABLE IF EXISTS tbl_terminal_value;
DROP TABLE IF EXISTS tbl_enterprise_value;
DROP TABLE IF EXISTS tbl_equity_value;
DROP TABLE IF EXISTS tbl_intrinsic_price;
DROP TABLE IF EXISTS tbl_intrinsic_price_driver;
DROP TABLE IF EXISTS tbl_final_valuation;

-- Any intermediate views
DROP VIEW IF EXISTS vw_latest_march;
""")

print("✅ All FCFF, growth, and valuation objects removed")


✅ All FCFF, growth, and valuation objects removed


In [489]:
#STEP 0 — LATEST MARCH SNAPSHOTS (MANDATORY)

In [581]:
con.execute("""
CREATE OR REPLACE VIEW vw_latest_march AS
SELECT *
FROM (
    SELECT *,
           ROW_NUMBER() OVER (
               PARTITION BY stock_id
               ORDER BY period_end DESC
           ) AS rn
    FROM income_statement_calculated
    WHERE EXTRACT(month FROM period_end) = 3
)
WHERE rn = 1;
""")

con.execute("""
CREATE OR REPLACE VIEW vw_latest_march_balance AS
SELECT *
FROM (
    SELECT *,
           ROW_NUMBER() OVER (
               PARTITION BY stock_id
               ORDER BY period_end DESC
           ) AS rn
    FROM balance_sheet_calculated
    WHERE EXTRACT(month FROM period_end) = 3
)
WHERE rn = 1;
""")


In [583]:
con.execute("""
SELECT sm.symbol, revenue, ebit, depreciation
FROM vw_latest_march v
JOIN stocks_master sm USING (stock_id)
WHERE sm.symbol IN ('RELIANCE','ASIANPAINT','BEL');
""").df()

,symbol,revenue,ebit,depreciation
0,RELIANCE,962820.0,112462.0,53136.0
1,BEL,23769.0,6370.0,467.0
2,ASIANPAINT,33906.0,4980.0,1026.0


In [584]:
#STEP 1 — REVENUE GROWTH (MAX OF 3Y, 5Y)

In [585]:
con.execute("""
DROP TABLE IF EXISTS tbl_revenue_growth;

CREATE TABLE tbl_revenue_growth AS
WITH base AS (
    SELECT
        stock_id,
        period_end,
        revenue,
        LAG(revenue, 3) OVER (PARTITION BY stock_id ORDER BY period_end) AS rev_3y,
        LAG(revenue, 5) OVER (PARTITION BY stock_id ORDER BY period_end) AS rev_5y
    FROM income_statement_calculated
    WHERE EXTRACT(month FROM period_end) = 3
),
calc AS (
    SELECT
        stock_id,
        POWER(revenue / rev_3y, 1.0/3) - 1 AS g_3y,
        POWER(revenue / rev_5y, 1.0/5) - 1 AS g_5y,
        ROW_NUMBER() OVER (PARTITION BY stock_id ORDER BY period_end DESC) AS rn
    FROM base
)
SELECT
    stock_id,
    GREATEST(g_3y, g_5y) AS revenue_growth
FROM calc
WHERE rn = 1;
""")


In [587]:
con.execute("""
SELECT sm.symbol, ROUND(revenue_growth*100,2) AS rev_g_pct
FROM tbl_revenue_growth g
JOIN stocks_master sm USING (stock_id)
WHERE sm.symbol IN ('RELIANCE','ASIANPAINT','BEL');
""").df()

,symbol,rev_g_pct
0,RELIANCE,11.49
1,BEL,15.65
2,ASIANPAINT,10.90


In [588]:
#STEP 2 — EBIT MARGIN (AVG 3Y)

In [589]:
con.execute("""
DROP TABLE IF EXISTS tbl_ebit_margin;

CREATE TABLE tbl_ebit_margin AS
WITH hist AS (
    SELECT
        stock_id,
        ebit / NULLIF(revenue, 0) AS margin,
        ROW_NUMBER() OVER (
            PARTITION BY stock_id
            ORDER BY period_end DESC
        ) AS rn
    FROM income_statement_calculated
    WHERE EXTRACT(month FROM period_end) = 3
)
SELECT
    stock_id,
    AVG(margin) FILTER (WHERE rn <= 3) AS ebit_margin
FROM hist
GROUP BY stock_id;
""")

print("✅ STEP 2: EBIT margin (3Y average) created")


✅ STEP 2: EBIT margin (3Y average) created


In [591]:
con.execute("""
SELECT
    sm.symbol,
    ROUND(ebit_margin * 100, 2) AS ebit_margin_pct
FROM tbl_ebit_margin m
JOIN stocks_master sm USING (stock_id)
WHERE sm.symbol IN ('RELIANCE', 'ASIANPAINT', 'BEL');
""").df()

,symbol,ebit_margin_pct
0,RELIANCE,11.91
1,BEL,23.39
2,ASIANPAINT,16.44


In [592]:
#STEP 3 — NET CAPEX INTENSITY (AVG)

In [594]:
con.execute("""
DROP TABLE IF EXISTS tbl_net_capex_intensity;

CREATE TABLE tbl_net_capex_intensity AS
WITH hist AS (
    SELECT
        isc.stock_id,
        ((-1 * cfc.capex) - isc.depreciation)
        / NULLIF(
            isc.revenue - LAG(isc.revenue) OVER (
                PARTITION BY isc.stock_id ORDER BY isc.period_end
            ),
            0
        ) AS capex_ratio,
        ROW_NUMBER() OVER (PARTITION BY isc.stock_id ORDER BY isc.period_end DESC) AS rn
    FROM income_statement_calculated isc
    JOIN cashflow_calculated cfc
      ON isc.stock_id = cfc.stock_id
     AND isc.period_end = cfc.period_end
    WHERE EXTRACT(month FROM isc.period_end) = 3
)
SELECT
    stock_id,
    AVG(capex_ratio) FILTER (WHERE rn <= 5) AS net_capex_intensity
FROM hist
GROUP BY stock_id;
""")


In [596]:
con.execute("""
SELECT sm.symbol, ROUND(net_capex_intensity,3)
FROM tbl_net_capex_intensity
JOIN stocks_master sm USING (stock_id)
WHERE sm.symbol IN ('RELIANCE','ASIANPAINT','BEL');
""").df()

,symbol,"round(net_capex_intensity, 3)"
0,RELIANCE,1.073
1,BEL,0.098
2,ASIANPAINT,0.168


In [597]:
#STEP 4 — WORKING CAPITAL INTENSITY (AVG)

In [598]:
con.execute("""
DROP TABLE IF EXISTS tbl_wc_intensity;

CREATE TABLE tbl_wc_intensity AS
WITH hist AS (
    SELECT
        isc.stock_id,
        cfc.working_capital_changes
        / NULLIF(
            isc.revenue - LAG(isc.revenue) OVER (
                PARTITION BY isc.stock_id ORDER BY isc.period_end
            ),
            0
        ) AS wc_ratio,
        ROW_NUMBER() OVER (PARTITION BY isc.stock_id ORDER BY isc.period_end DESC) AS rn
    FROM income_statement_calculated isc
    JOIN cashflow_calculated cfc
      ON isc.stock_id = cfc.stock_id
     AND isc.period_end = cfc.period_end
    WHERE EXTRACT(month FROM isc.period_end) = 3
)
SELECT
    stock_id,
    AVG(wc_ratio) FILTER (WHERE rn <= 5) AS wc_intensity
FROM hist
GROUP BY stock_id;
""")


In [600]:
con.execute("""
SELECT sm.symbol, ROUND(wc_intensity,3)
FROM tbl_wc_intensity
JOIN stocks_master sm USING (stock_id)
WHERE sm.symbol IN ('RELIANCE','ASIANPAINT','BEL');
""").df()

,symbol,"round(wc_intensity, 3)"
0,RELIANCE,0.186
1,BEL,0.161
2,ASIANPAINT,-0.058


In [601]:
#STEP 5 — REVENUE FORECAST (7 YEARS)

In [602]:
con.execute("""
DROP TABLE IF EXISTS tbl_revenue_forecast;

CREATE TABLE tbl_revenue_forecast AS
SELECT
    v.stock_id,
    yr,
    v.revenue * POWER(1 + g.revenue_growth, yr) AS revenue_t
FROM vw_latest_march v
JOIN tbl_revenue_growth g USING (stock_id)
CROSS JOIN (SELECT UNNEST([1,2,3,4,5,6,7]) AS yr);
""")


In [603]:
con.execute("""
SELECT sm.symbol, yr, ROUND(revenue_t,0)
FROM tbl_revenue_forecast
JOIN stocks_master sm USING (stock_id)
WHERE sm.symbol IN ('RELIANCE','ASIANPAINTS','BEL');
""").df()

,symbol,yr,"round(revenue_t, 0)"
0,RELIANCE,7,2062193.0
1,BEL,7,65755.0
2,RELIANCE,6,1849586.0
3,BEL,6,56859.0
4,RELIANCE,5,1658898.0
5,BEL,5,49166.0
6,RELIANCE,4,1487869.0
7,BEL,4,42514.0
8,RELIANCE,3,1334473.0
9,BEL,3,36762.0


In [604]:
#STEP 6 — FCFF (FINAL, CORRECT)

In [605]:
con.execute("""
DROP TABLE IF EXISTS tbl_fcff_forecast;

CREATE TABLE tbl_fcff_forecast AS
WITH base AS (
    SELECT
        r.stock_id,
        r.yr,
        r.revenue_t,
        v.revenue AS revenue_0,
        m.ebit_margin,
        v.tax_expense_percent / 100.0 AS tax_rate,
        c.net_capex_intensity,
        w.wc_intensity,
        LAG(r.revenue_t) OVER (PARTITION BY r.stock_id ORDER BY r.yr) AS prev_rev
    FROM tbl_revenue_forecast r
    JOIN vw_latest_march v USING (stock_id)
    JOIN tbl_ebit_margin m USING (stock_id)
    JOIN tbl_net_capex_intensity c USING (stock_id)
    JOIN tbl_wc_intensity w USING (stock_id)
)
SELECT
    stock_id,
    yr,

    -- NOPAT
    revenue_t * ebit_margin * (1 - tax_rate) AS nopat,

    -- Incremental CAPEX (declining)
    (
        CASE WHEN yr = 1 THEN revenue_t - revenue_0
             ELSE revenue_t - prev_rev
        END
    ) * net_capex_intensity
      * (0.75 - (yr - 1)*(0.75-0.30)/6) AS incr_capex,

    -- Incremental WC
    (
        CASE WHEN yr = 1 THEN revenue_t - revenue_0
             ELSE revenue_t - prev_rev
        END
    ) * wc_intensity AS delta_wc,

    -- FCFF
    (revenue_t * ebit_margin * (1 - tax_rate))
    - (
        (CASE WHEN yr = 1 THEN revenue_t - revenue_0
              ELSE revenue_t - prev_rev
         END) * net_capex_intensity
         * (0.75 - (yr - 1)*(0.75-0.30)/6)
      )
    + (
        (CASE WHEN yr = 1 THEN revenue_t - revenue_0
              ELSE revenue_t - prev_rev
         END) * wc_intensity
      ) AS fcff

FROM base;
""")


In [608]:
con.execute("""
SELECT sm.symbol, yr, ROUND(fcff,0)
FROM tbl_fcff_forecast
JOIN stocks_master sm USING (stock_id)
WHERE sm.symbol IN ('RELIANCE','ASIANPAINT','BEL');
""").df()

,symbol,yr,"round(fcff, 0)"
0,BEL,1,5086.0
1,BEL,2,5914.0
2,BEL,3,6875.0
3,BEL,4,7993.0
4,BEL,5,9293.0
5,BEL,6,10803.0
6,BEL,7,12558.0
7,ASIANPAINT,1,3833.0
8,ASIANPAINT,2,4302.0
9,ASIANPAINT,3,4829.0


In [612]:
#STEP 7A — Create tbl_terminal_growth

In [642]:
con.execute("""
DROP TABLE IF EXISTS tbl_terminal_growth;

CREATE TABLE tbl_terminal_growth AS
SELECT
    stock_id,
    CASE
        WHEN sector IN ('Consumer Durables', 'Consumer Staples', 'Paints')
            THEN 0.04
        WHEN sector IN ('Capital Goods', 'Defence')
            THEN 0.04
        WHEN sector IN ('Energy', 'Oil & Gas', 'Conglomerates')
            THEN 0.05
        WHEN sector IN ('Information Technology', 'IT Services')
            THEN 0.05
        WHEN sector IN ('Utilities')
            THEN 0.025
        ELSE 0.03
    END AS terminal_growth
FROM stocks_master;
""")


In [643]:
con.execute("""
SELECT
    sm.symbol,
    sm.sector,
    tg.terminal_growth * 100 AS terminal_g_pct
FROM tbl_terminal_growth tg
JOIN stocks_master sm
  ON tg.stock_id = sm.stock_id
WHERE sm.symbol IN ('RELIANCE','ASIANPAINT','BEL');
""").df()

,symbol,sector,terminal_g_pct
0,RELIANCE,Energy,5.0
1,BEL,Information Technology,5.0
2,ASIANPAINT,Materials,3.0


In [619]:
#STEP 7B — TERMINAL VALUE (GUARDED)

In [644]:
con.execute("""
DROP TABLE IF EXISTS tbl_terminal_value;

CREATE TABLE tbl_terminal_value AS
SELECT
    f.stock_id,

    -- growth used after guard
    CASE
        WHEN tg.terminal_growth >= w.wacc
        THEN w.wacc - 0.02
        ELSE tg.terminal_growth
    END AS terminal_growth_used,

    -- terminal value
    f.fcff
    * (1 + CASE
                WHEN tg.terminal_growth >= w.wacc
                THEN w.wacc - 0.02
                ELSE tg.terminal_growth
           END)
    / NULLIF(
        w.wacc -
        CASE
            WHEN tg.terminal_growth >= w.wacc
            THEN w.wacc - 0.02
            ELSE tg.terminal_growth
        END,
        0
    ) AS terminal_value

FROM tbl_fcff_forecast f
JOIN vw_wacc_latest w
  ON f.stock_id = w.stock_id
JOIN tbl_terminal_growth tg
  ON f.stock_id = tg.stock_id
WHERE f.yr = 7;
""")

print("✅ STEP 7B: Terminal value created")


✅ STEP 7B: Terminal value created


In [645]:
con.execute("""
SELECT
    sm.symbol,
    ROUND(terminal_growth_used*100,2) AS terminal_g_used_pct,
    ROUND(terminal_value,0) AS terminal_value_cr
FROM tbl_terminal_value tv
JOIN stocks_master sm USING (stock_id)
WHERE sm.symbol IN ('RELIANCE','ASIANPAINT','BEL');
""").df()

,symbol,terminal_g_used_pct,terminal_value_cr
0,ASIANPAINT,3.0,107657.0
1,RELIANCE,5.0,2572972.0
2,BEL,5.0,97188.0


In [623]:
#STEP 8A — PV of FCFF (Years 1–7)

In [646]:
con.execute("""
DROP TABLE IF EXISTS tbl_fcff_pv;

CREATE TABLE tbl_fcff_pv AS
SELECT
    f.stock_id,
    SUM(
        f.fcff / POWER(1 + w.wacc, f.yr)
    ) AS pv_fcff
FROM tbl_fcff_forecast f
JOIN vw_wacc_latest w
  ON f.stock_id = w.stock_id
GROUP BY f.stock_id;
""")

print("✅ STEP 8A: PV of FCFF created")


✅ STEP 8A: PV of FCFF created


In [647]:
con.execute("""
SELECT
    sm.symbol,
    ROUND(pv_fcff,0) AS pv_fcff_cr
FROM tbl_fcff_pv
JOIN stocks_master sm USING (stock_id)
WHERE sm.symbol IN ('RELIANCE','ASIANPAINT','BEL');
""").df()

,symbol,pv_fcff_cr
0,RELIANCE,349142.0
1,BEL,28331.0
2,ASIANPAINT,25615.0


In [627]:
#STEP 8B — PV of Terminal Value

In [648]:
con.execute("""
DROP TABLE IF EXISTS tbl_terminal_value_pv;

CREATE TABLE tbl_terminal_value_pv AS
SELECT
    tv.stock_id,
    tv.terminal_value / POWER(1 + w.wacc, 7) AS pv_terminal_value
FROM tbl_terminal_value tv
JOIN vw_wacc_latest w
  ON tv.stock_id = w.stock_id;
""")

print("✅ STEP 8B: PV of terminal value created")


✅ STEP 8B: PV of terminal value created


In [629]:
#STEP 8C — Enterprise Value

In [649]:
con.execute("""
DROP TABLE IF EXISTS tbl_enterprise_value;

CREATE TABLE tbl_enterprise_value AS
SELECT
    f.stock_id,
    f.pv_fcff + t.pv_terminal_value AS enterprise_value
FROM tbl_fcff_pv f
JOIN tbl_terminal_value_pv t
  ON f.stock_id = t.stock_id;
""")

print("✅ STEP 8C: Enterprise value created")


✅ STEP 8C: Enterprise value created


In [650]:
con.execute("""
SELECT
    sm.symbol,
    ROUND(enterprise_value,0) AS ev_cr
FROM tbl_enterprise_value
JOIN stocks_master sm USING (stock_id)
WHERE sm.symbol IN ('RELIANCE','ASIANPAINT','BEL');
""").df()

,symbol,ev_cr
0,RELIANCE,1554264.0
1,BEL,57833.0
2,ASIANPAINT,79751.0


In [633]:
#STEP 8D — Equity Value

In [651]:
con.execute("""
DROP TABLE IF EXISTS tbl_equity_value;

CREATE TABLE tbl_equity_value AS
SELECT
    e.stock_id,
    e.enterprise_value - b.net_debt AS equity_value
FROM tbl_enterprise_value e
JOIN vw_latest_march_balance b
  ON e.stock_id = b.stock_id;
""")

print("✅ STEP 8D: Equity value created")


✅ STEP 8D: Equity value created


In [652]:
con.execute("""
SELECT
    sm.symbol,
    ROUND(equity_value,0) AS equity_value_cr
FROM tbl_equity_value
JOIN stocks_master sm USING (stock_id)
WHERE sm.symbol IN ('RELIANCE','ASIANPAINT','BEL');
""").df()

,symbol,equity_value_cr
0,RELIANCE,1286453.0
1,BEL,67317.0
2,ASIANPAINT,78243.0


In [653]:
con.execute("""
DROP TABLE IF EXISTS tbl_intrinsic_price;

CREATE TABLE tbl_intrinsic_price AS
SELECT
    e.stock_id,
    (e.equity_value * 1e7) / v.shares_outstanding AS intrinsic_price
FROM tbl_equity_value e
JOIN vw_latest_march v
  ON e.stock_id = v.stock_id;
""")

print("✅ STEP 8E: Intrinsic price created")


✅ STEP 8E: Intrinsic price created


In [654]:
con.execute("""
SELECT
    sm.symbol,
    ROUND(ip.intrinsic_price, 2) AS intrinsic_price_rs,
    ROUND(v.shares_outstanding, 2) AS shares_outstanding
FROM tbl_intrinsic_price ip
JOIN stocks_master sm
  ON ip.stock_id = sm.stock_id
JOIN vw_latest_march v
  ON ip.stock_id = v.stock_id
WHERE sm.symbol IN ('RELIANCE','ASIANPAINT','BEL');

""").df()

,symbol,intrinsic_price_rs,shares_outstanding
0,RELIANCE,950.69,1.353177e+10
1,BEL,92.10,7.309066e+09
2,ASIANPAINT,815.71,9.591943e+08
